# Apprentice network and the human-like solver

An apprentice network learns which moves are good from the solver's own experience, by expert iteration (`rl.expert_iteration`). Round 1 learns from plain AND-OR's solved attempts; each later round from the human-like solver the network has become. From each solved attempt it learns the moves played (loops cut out: a return to an earlier board rewinds to it) and, at every subgoal of every plan that worked, which option resolved it.

The **human-like solver** then arbitrates between intuition (the network's move) and deliberation (AND-OR's subgoal-by-subgoal reasoning, ordered by the network, with lapses `GAMMA`).

Training on many *different* puzzles is what makes the network transfer (`scripts/learnability.py`).

In [1]:
import sys
sys.path.append("../scripts")
import os
import random
import statistics

import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

import and_or as ao
import rl

## Parameters

`N_TRAIN_PER_BIN = 200` gives 1,000 training puzzles. Each of the `EXIT_ROUNDS` rounds makes `ATTEMPTS_PER_PUZZLE` attempts per training and validation puzzle, then trains.

`HEURISTIC_TEMPERATURE` adds noise to the network's ordering, so options are tried in an order sampled by its preference rather than always the same one (see `rl.PolicyAgent`). At 0 the same board always gets the same choice and the solver repeats plans that undo each other; in a test at gamma 0.07, 1.0 raised the solve rate on 13-20-move puzzles by about 20 points over 0. It also applies while the human-like solver makes attempts in expert-iteration rounds 2+.

`ARBITRATION_THRESHOLDS` and `MEMORY` set the solver (see "Solving held-out puzzles" below).

In [2]:
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
BINS = [(1, 8), (9, 12), (13, 16), (17, 20), (21, 30)]  # distance-to-goal bins, sampled evenly
N_TRAIN_PER_BIN = 200
N_VAL_PER_BIN = 20  # validation puzzles: pick the best epoch
N_TEST_PER_BIN = 20  # held-out puzzles: never trained or selected on
HEURISTIC_TEMPERATURE = 1.0

# expert iteration
EXIT_ROUNDS = 3
ATTEMPTS_PER_PUZZLE = 4
ATTEMPT_GAMMA = 0.07  # lapse probability while making training attempts

# evaluation
STATES_PER_PUZZLE = 50  # boards per test puzzle scored against exact search labels
SUBGOALS_PER_PUZZLE = 200  # subgoals per test puzzle scored against the oracle
GAMMAS = [0.07, 0.2]  # AND-OR lapse probabilities to compare
ARBITRATION_THRESHOLDS = [0.0, 0.5, 0.8, 1.01]  # intuition's confidence bar; 0 = intuition only, > 1 = deliberation only
MEMORY = 10  # recent boards the solver remembers; a move back to one of them triggers deliberation
N_RUNS = 3  # solve attempts per held-out puzzle and solver
MAX_STEPS = 200

## Loading data

Test, validation and training puzzles are disjoint samples from `rush_nw.txt`, evenly spread over the distance bins. The test puzzles get exact search labels - a yardstick for evaluation only; the network never learns from them.

In [3]:
pool = rl.load_puzzle_pool("../data/rush_nw.txt")
test = rl.sample_by_distance(pool, BINS, N_TEST_PER_BIN, seed=SEED)
val = rl.sample_by_distance(pool, BINS, N_VAL_PER_BIN, seed=SEED + 1, exclude=[s for _, s in test])
train = rl.sample_by_distance(pool, BINS, N_TRAIN_PER_BIN, seed=SEED + 2,
                              exclude=[s for _, s in test + val])
print(f"{len(train)} training, {len(val)} validation, {len(test)} test puzzles")

test_labels = rl.label_puzzles([s for _, s in test], STATES_PER_PUZZLE, seed=SEED + 2)
test_subgoals = rl.label_subgoals([s for _, s in test], SUBGOALS_PER_PUZZLE, seed=SEED + 2)

1000 training, 100 validation, 100 test puzzles


labelling subgoals: 100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


## Training the apprentice

Three heads share one network (see `rl.PolicyNetwork`): which legal move to play now; for every car and destination, how soon it should get there - which, trained on subgoal choices too, is what orders AND-OR's options; and the distance to the goal.

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
agent = rl.PolicyAgent(device=DEVICE, temperature=HEURISTIC_TEMPERATURE)

rounds = rl.expert_iteration(agent, [s for _, s in train], [s for _, s in val], rounds=EXIT_ROUNDS,
                             n_attempts=ATTEMPTS_PER_PUZZLE, gamma=ATTEMPT_GAMMA, seed=SEED)
for r in rounds:
    print(f"round {r['round']}: {r['solved']}/{r['attempts']} training attempts solved, "
          f"{r['boards']} boards and {r['subgoals']} subgoals learned from; validation agreement with "
          f"the round's own attempts: moves {r['agreement']:.0%}, subgoal options {r['subgoal_agreement']:.0%}")

attempts:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
os.makedirs("../models", exist_ok=True)
agent.save("../models/policy_agent_attempts.pt")
# To reuse a trained agent instead of retraining:
# agent = rl.PolicyAgent.load("../models/policy_agent_attempts.pt", device=DEVICE)
# agent.temperature = HEURISTIC_TEMPERATURE

## Held-out agreement

Against exact search labels on the test puzzles, by distance bin:

- **move**: how often the network's top legal move is optimal (vs. a random legal move);
- **ordering**: at the subgoals an oracle-ordered AND-OR search visits, how often the network's top option is one an optimal solution prefers (vs. a random option) - the judgment the human-like solver uses;
- **distance error**: how far off its distance-to-goal estimate is.

In [ ]:
for (lo, hi), start in zip(BINS, range(0, len(test), N_TEST_PER_BIN)):
    boards = [e for ex in test_labels[start:start + N_TEST_PER_BIN] for e in ex]
    subgoals = [g for gs in test_subgoals[start:start + N_TEST_PER_BIN] for g in gs]
    m, o = rl.evaluate_agreement(agent, boards), rl.evaluate_subgoals(agent, subgoals)
    print(f"distance {lo:>2}-{hi:<2}: move {m['agreement']:.0%} (random {m['chance']:.0%}), "
          f"ordering {o['agreement']:.0%} (random {o['chance']:.0%}), distance error {m['distance_error']:.1f} moves")

## Solving held-out puzzles

The human-like solver arbitrates every turn between two modes (`rl.solve_arbitrated`):

- **intuition** - play the network's top legal move, when its probability is at least the threshold and it doesn't return to one of the last `MEMORY` boards;
- **deliberation** - otherwise, one AND-OR pass ordered by the network (lapse probability `gamma`), whose moves are played. Going in circles counts as low confidence: a move back to a recent board makes the solver stop and think.

Threshold 0 is intuition only (apart from revisits); above 1 it is deliberation only. In between, deliberation is kept for the boards where intuition is unsure - as learning raises confidence, the solver should rely on intuition more and think less. Plain AND-OR (random ordering, random lapse moves) is the untrained reference.

Each panel is one lapse probability. Rows: solve rate; moves as a multiple of the optimal, over solved attempts; and the share of moves made by intuition.

In [ ]:
def bin_of(distance):
    return next((lo, hi) for lo, hi in BINS if lo <= distance <= hi)


def summarize(runs):
    """{bin: (solve rate, mean moves / optimal over solved, share of moves by intuition,
    deliberations per attempt)} from a list of attempt dicts."""
    out = {}
    for b in BINS:
        rs = [r for r in runs if bin_of(r["distance"]) == b]
        ratios = [r["moves"] / r["distance"] for r in rs if r["solved"]]
        played = sum(r.get("intuitive", 0) + r.get("deliberate", 0) for r in rs)
        out[b] = (sum(r["solved"] for r in rs) / len(rs),
                  statistics.mean(ratios) if ratios else float("nan"),
                  sum(r.get("intuitive", 0) for r in rs) / played if played else float("nan"),
                  statistics.mean(r.get("deliberations", 0) for r in rs))
    return out


def solver_name(threshold):
    if threshold <= 0:
        return "intuition only"
    if threshold > 1:
        return "deliberation only"
    return f"threshold {threshold}"


results = {}  # (solver, gamma) -> summarize(...)
for gamma in GAMMAS:
    random.seed(SEED)
    plain = []
    for d, state in tqdm(test, desc=f"plain AND-OR, gamma={gamma}"):
        for _ in range(N_RUNS):
            moves = rl.solve_and_or(state, MAX_STEPS, gamma=gamma)
            plain.append({"solved": moves is not None, "moves": len(moves) if moves is not None else None,
                          "distance": d})
    results[("plain AND-OR", gamma)] = summarize(plain)
    for threshold in ARBITRATION_THRESHOLDS:
        runs = rl.evaluate_arbitrated(agent, test, N_RUNS, gamma, threshold, MEMORY, MAX_STEPS, seed=SEED)
        results[(solver_name(threshold), gamma)] = summarize(runs)

for (solver, gamma), by_bin in results.items():
    print(f"{solver}, gamma={gamma}")
    for (lo, hi), (rate, ratio, share, deliberations) in by_bin.items():
        extra = "" if solver == "plain AND-OR" else \
            f", {share:.0%} of moves by intuition, {deliberations:.1f} deliberations per attempt"
        print(f"  distance {lo:>2}-{hi:<2}: solved {rate:.0%}, {ratio:.1f}x optimal{extra}")

In [ ]:
labels = [f"{lo}-{hi}" for lo, hi in BINS]
solvers = ["plain AND-OR"] + [solver_name(t) for t in ARBITRATION_THRESHOLDS]
fig, axes = plt.subplots(3, len(GAMMAS), figsize=(6 * len(GAMMAS), 11), sharex=True, squeeze=False)
for col, gamma in enumerate(GAMMAS):
    for row, (ylabel, k) in enumerate((("Solve rate", 0), ("Moves / optimal (solved attempts)", 1),
                                       ("Share of moves by intuition", 2))):
        ax = axes[row][col]
        for solver in solvers:
            if solver == "plain AND-OR" and k == 2:
                continue
            ax.plot(labels, [results[(solver, gamma)][b][k] for b in BINS], marker="o", label=solver)
        ax.set_ylabel(ylabel)
        if k != 1:
            ax.set_ylim(0, 1.05)
        else:
            ax.axhline(1.0, color="gray", linestyle="--")
    axes[0][col].set_title(f"gamma = {gamma}")
    axes[-1][col].set_xlabel("Distance to goal")
axes[0][0].legend()
plt.tight_layout()
os.makedirs("../visualization", exist_ok=True)
plt.savefig("../visualization/arbitration.png")
plt.show()